# Factorised Latent-Action World Model in Colab

This notebook installs the repo, generates passive and labelled datasets, trains the main baselines, evaluates them, plots the results, and exports the artifacts.


## Before you run

1. Push this repo to GitHub.
2. In Colab, switch runtime to GPU.
3. Edit `REPO_URL` in the next cell.
4. Leave `SLOT_ENTITY_WEIGHT=1.0` for a practical slot variant, or set it to `0.0` if you want a stricter pixel-only run.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/dhruvsyam123/factorized-latent-world-models.git"
BRANCH = "main"
WORKDIR = Path("/content/factorise-latent-wm")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(WORKDIR)], check=True)

os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "uv"], check=True)
env = os.environ.copy()
env["UV_CACHE_DIR"] = str(WORKDIR / ".uv-cache")
subprocess.run(["uv", "pip", "install", "--system", "-e", ".[dev,colab]"], check=True, env=env)
print(f"Setup complete in {WORKDIR}")

import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path

RUN_ROOT = Path("colab_runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)

PASSIVE_TRANSITIONS = 10000
LABELLED_TRANSITIONS = [1000, 2500]
STAGE1_EPOCHS = 6
STAGE2_EPOCHS = 3
BATCH_SIZE = 128
EVAL_EPISODES = 10
PLANNER_HORIZON = 4
DEVICE = "cuda"

# Mainline: state_factor. Optional extension: pixel_slot.
RUN_SLOT_VARIANT = False
SLOT_ENTITY_WEIGHT = 1.0
print("Configured run root:", RUN_ROOT)


In [ ]:
import json
import shlex
import subprocess

import matplotlib.pyplot as plt
import pandas as pd


def run_cmd(cmd):
    print("$", shlex.join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    completed.check_returncode()
    return completed.stdout.strip()


def json_cmd(cmd):
    output = run_cmd(cmd)
    start = output.find("{")
    end = output.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"Command did not return JSON: {output}")
    return json.loads(output[start : end + 1])


def train_variant(run_dir, variant, labelled_transitions):
    name = variant["name"]
    checkpoints = run_dir / "checkpoints"
    checkpoints.mkdir(parents=True, exist_ok=True)
    data_dir = run_dir / "data"
    train_split = "labelled_train.npz" if variant["baseline"] == "action" else "passive_train.npz"
    val_split = "labelled_val.npz" if variant["baseline"] == "action" else "passive_val.npz"
    stage1_ckpt = checkpoints / f"{name}_stage1.pt"
    final_ckpt = stage1_ckpt

    cmd = [
        "flwm", "train-stage1",
        "--baseline", variant["baseline"],
        "--train", str(data_dir / train_split),
        "--val", str(data_dir / val_split),
        "--output", str(stage1_ckpt),
        "--epochs", str(STAGE1_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
    ]
    if variant.get("encoder_type") is not None:
        cmd += ["--encoder-type", variant["encoder_type"]]
    if variant.get("entity_weight") is not None:
        cmd += ["--entity-weight", str(variant["entity_weight"])]
    run_cmd(cmd)

    if variant.get("needs_stage2", False):
        final_ckpt = checkpoints / f"{name}_stage2.pt"
        run_cmd([
            "flwm", "train-stage2",
            "--checkpoint", str(stage1_ckpt),
            "--labelled", str(data_dir / "labelled_train.npz"),
            "--output", str(final_ckpt),
            "--epochs", str(STAGE2_EPOCHS),
            "--batch-size", str(BATCH_SIZE),
            "--device", DEVICE,
        ])

    metrics = json_cmd([
        "flwm", "evaluate",
        "--checkpoint", str(final_ckpt),
        "--episodes", str(EVAL_EPISODES),
        "--horizon", str(PLANNER_HORIZON),
        "--device", DEVICE,
    ])
    return {
        "variant": name,
        "baseline": variant["baseline"],
        "encoder_type": variant.get("encoder_type", "n/a"),
        "labelled_transitions": labelled_transitions,
        "checkpoint": str(final_ckpt),
        **metrics,
    }


In [ ]:
variants = [
    {"name": "factor_state", "baseline": "factor", "encoder_type": "state_factor", "needs_stage2": True},
    {"name": "monolithic", "baseline": "monolithic", "needs_stage2": True},
    {"name": "action_conditioned", "baseline": "action", "needs_stage2": False},
]
if RUN_SLOT_VARIANT:
    variants.insert(1, {
        "name": "factor_pixel_slot",
        "baseline": "factor",
        "encoder_type": "pixel_slot",
        "entity_weight": SLOT_ENTITY_WEIGHT,
        "needs_stage2": True,
    })

results = []
for labelled in LABELLED_TRANSITIONS:
    run_dir = RUN_ROOT / f"labelled_{labelled}"
    data_dir = run_dir / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    json_cmd([
        "flwm", "generate-data",
        "--output-dir", str(data_dir),
        "--passive-transitions", str(PASSIVE_TRANSITIONS),
        "--labelled-transitions", str(labelled),
    ])

    for variant in variants:
        results.append(train_variant(run_dir, variant, labelled))

results_df = pd.DataFrame(results).sort_values(["variant", "labelled_transitions"])
results_df.to_csv(RUN_ROOT / "results.csv", index=False)
results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for variant, frame in results_df.groupby("variant"):
    axes[0].plot(frame["labelled_transitions"], frame["success_rate"], marker="o", label=variant)
    axes[1].plot(frame["labelled_transitions"], frame["mean_reward"], marker="o", label=variant)

axes[0].set_title("Success Rate vs Labelled Data")
axes[0].set_xlabel("Labelled transitions")
axes[0].set_ylabel("Success rate")
axes[0].legend()

axes[1].set_title("Mean Reward vs Labelled Data")
axes[1].set_xlabel("Labelled transitions")
axes[1].set_ylabel("Mean reward")
axes[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
import shutil

archive_path = shutil.make_archive("factor_latent_wm_colab_artifacts", "zip", RUN_ROOT)
print("Wrote artifact archive:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception as exc:
    print("Auto-download skipped:", exc)
